# EX11 — Optimisation de R2
Commande anticipative avec frottements et PID distincts. Seuils pédagogiques : RMS ≤ 10 mm, maximum ≤ 20 mm, saisie/dépose ≤ 10 mm. Recherche discrète de 72 configurations, priorité à la durée puis à la RMS.

Le profil quintique $f(u)=10u^3-15u^4+6u^5$ satisfait $f(0)=0,f(1)=1$ et annule les deux premières dérivées aux extrémités. La commande ajoute $B\dot q_r$ au couple anticipé et les corrections PID. Les limites de couple incluent le facteur 1,5.

In [1]:
import sys, types

m=types.ModuleType('support_s2'); sys.modules['support_s2']=m
exec('"""Support fourni : cinématique et mission S2. Interface en mètres, angles en degrés."""\n\nimport math\n\ndef est_accessible(L1, L2, x, y):\n    if not all((math.isfinite(v) for v in (L1, L2, x, y))) or min(L1, L2) <= 0 or L1 == L2:\n        raise ValueError(\'Longueurs positives distinctes et valeurs finies requises\')\n    d = math.hypot(x, y)\n    return abs(L1 - L2) - 1e-10 <= d <= L1 + L2 + 1e-10\n\ndef angles_coude(L1, L2, x, y):\n    if not est_accessible(L1, L2, x, y):\n        raise ValueError(\'Cible inaccessible\')\n    c = (x * x + y * y - L1 * L1 - L2 * L2) / (2 * L1 * L2)\n    c = max(-1.0, min(1.0, c))\n    b = math.degrees(math.acos(c))\n    return (b, -b)\n\ndef angle_epaule(L1, L2, x, y, theta2):\n    b = math.radians(theta2)\n    return math.degrees(math.atan2(y, x) - math.atan2(L2 * math.sin(b), L1 + L2 * math.cos(b)))\n\ndef normaliser(angle):\n    return (angle + 180) % 360 - 180\n\ndef cinematique_inverse(L1, L2, x, y):\n    solutions = []\n    for b in angles_coude(L1, L2, x, y):\n        a = angle_epaule(L1, L2, x, y, b)\n        q = (normaliser(a), normaliser(b))\n        if not any((all((abs(normaliser(v - w)) <= 1e-07 for v, w in zip(q, s))) for s in solutions)):\n            solutions.append(q)\n    return solutions\n\ndef ecart_angle(a, b):\n    return (b - a + 180) % 360 - 180\n\ndef cout(depart, arrivee):\n    return sum((abs(ecart_angle(a, b)) for a, b in zip(depart, arrivee)))\n\ndef choisir_solution(solutions, depart):\n    if not solutions:\n        raise ValueError(\'Aucune solution\')\n    return min(solutions, key=lambda q: cout(depart, q))\n\ndef segment_accessible(L1, L2, depart, arrivee):\n    if not (est_accessible(L1, L2, *depart) and est_accessible(L1, L2, *arrivee)):\n        return False\n    x, y = depart\n    u, v = (arrivee[0] - x, arrivee[1] - y)\n    norme2 = u * u + v * v\n    t = max(0, min(1, -(x * u + y * v) / norme2)) if norme2 else 0\n    return math.hypot(x + t * u, y + t * v) >= abs(L1 - L2) - 1e-10\n\ndef points_segment(depart, arrivee, N):\n    if type(N) is not int or N <= 0:\n        raise ValueError(\'N entier positif requis\')\n    return [(depart[0] + k / N * (arrivee[0] - depart[0]), depart[1] + k / N * (arrivee[1] - depart[1])) for k in range(N + 1)]\n\ndef planifier(L1, L2, depart, arrivee, N, angles_depart):\n    if not segment_accessible(L1, L2, depart, arrivee):\n        raise ValueError(\'Segment inaccessible\')\n    precedent = angles_depart\n    configurations = []\n    for x, y in points_segment(depart, arrivee, N):\n        q = choisir_solution(cinematique_inverse(L1, L2, x, y), precedent)\n        precedent = tuple((a + ecart_angle(a, b) for a, b in zip(precedent, q)))\n        configurations.append(precedent)\n    return configurations\n\nMISSION = [{\'cible\': (60, 0), \'action\': \'attente\'}, {\'cible\': (50, 20), \'action\': \'approche\'}, {\'cible\': (50, 10), \'action\': \'saisie\'}, {\'cible\': (50, 30), \'action\': \'degagement\'}, {\'cible\': (20, 50), \'action\': \'transfert\'}, {\'cible\': (20, 30), \'action\': \'depose\'}, {\'cible\': (20, 50), \'action\': \'degagement\'}, {\'cible\': (60, 0), \'action\': \'attente\'}]\n\ndef preparer_mission(L1, L2, mission, N=20):\n    if not mission or type(N) is not int or N <= 0:\n        raise ValueError(\'Mission ou N invalide\')\n    depart = mission[0][\'cible\']\n    q = cinematique_inverse(L1, L2, *depart)[0]\n    etat = \'ouverte\'\n    images = [(*q, etat)]\n    for etape in mission[1:]:\n        arrivee = etape[\'cible\']\n        trajet = planifier(L1, L2, depart, arrivee, N, q)\n        for i, q in enumerate(trajet[1:], 1):\n            if i == N:\n                if etape[\'action\'] == \'saisie\':\n                    etat = \'fermee\'\n                elif etape[\'action\'] == \'depose\':\n                    etat = \'ouverte\'\n            images.append((*q, etat))\n        depart = arrivee\n    return images\n\n_preparer_mission = preparer_mission\nMISSION = [dict(e, cible=tuple(v/100 for v in e[\'cible\'])) for e in MISSION]\ndef preparer_mission():\n    return _preparer_mission(.4,.3,MISSION,20)\n',m.__dict__)

m=types.ModuleType('support_s3'); sys.modules['support_s3']=m
exec('"""Support S3 autonome : angles en degrés (interface historique)."""\nimport math\nROBOT={\'L1\':.4,\'L2\':.3,\'m1\':1.2,\'m2\':.8,\'masse_pince\':.2,\'couple_max_1\':20.,\'couple_max_2\':8.,\'securite\':1.5}\ndef calculer_couples(robot,theta1,theta2,charge):\n    if not math.isfinite(charge) or charge < 0: raise ValueError(\'Charge invalide\')\n    a,b=map(math.radians,(theta1,theta2))\n    L1,L2,m1,m2,mp=(robot[k] for k in (\'L1\',\'L2\',\'m1\',\'m2\',\'masse_pince\'))\n    m=mp+charge\n    c2=9.81*(m2/2+m)*L2*math.cos(a+b)\n    return 9.81*(m1/2+m2+m)*L1*math.cos(a)+c2,c2\n\ndef sollicitations(robot,theta1,theta2,charge):\n    c=calculer_couples(robot,theta1,theta2,charge)\n    return tuple(robot[\'securite\']*abs(v)/robot[f\'couple_max_{i}\'] for i,v in enumerate(c,1))\ndef admissible(robot,theta1,theta2,charge):\n    return max(sollicitations(robot,theta1,theta2,charge))<=1\n',m.__dict__)

m=types.ModuleType('support_s4'); sys.modules['support_s4']=m
exec('"""Support S4 et moteur de simulation fourni pour S5. SI, angles en radians."""\nimport math\nfrom support_s2 import MISSION, cinematique_inverse, choisir_solution, ecart_angle\nfrom support_s3 import ROBOT, calculer_couples\n\ndef profil_cubique(theta_depart, theta_fin, T, t):\n    if T<=0 or not 0<=t<=T: raise ValueError(\'Temps invalide\')\n    u=t/T\n    d=theta_fin-theta_depart\n    return theta_depart+d*(3*u*u-2*u**3), d/T*(6*u-6*u*u), d/T**2*(6-12*u)\n\n\ndef coefficients(robot,charge):\n    L1,L2,m1,m2,mp=(robot[k] for k in (\'L1\',\'L2\',\'m1\',\'m2\',\'masse_pince\'))\n    m=mp+charge\n    return (m1/3+m2+m)*L1**2,(m2/3+m)*L2**2,(m2/2+m)*L1*L2\n\ndef termes_dynamiques(robot,q,w,charge):\n    A,B,D=coefficients(robot,charge)\n    b=q[1]; w1,w2=w\n    M=[[A+B+2*D*math.cos(b),B+D*math.cos(b)],[B+D*math.cos(b),B]]\n    h=(-D*math.sin(b)*(2*w1*w2+w2*w2),D*math.sin(b)*w1*w1)\n    G=calculer_couples(robot,*map(math.degrees,q),charge)\n    return M,h,G\n\ndef couples(robot,q,w,a,charge):\n    M,h,G=termes_dynamiques(robot,q,w,charge)\n    return tuple(sum(x*y for x,y in zip(row,a))+hi+gi for row,hi,gi in zip(M,h,G))\n\n\ndef postures():\n    q=cinematique_inverse(.4,.3,*MISSION[0][\'cible\'])[0]\n    result=[tuple(map(math.radians,q))]\n    for etape in MISSION[1:]:\n        cible=choisir_solution(cinematique_inverse(.4,.3,*etape[\'cible\']),q)\n        q=tuple(a+ecart_angle(a,b) for a,b in zip(q,cible))\n        result.append(tuple(map(math.radians,q)))\n    return result\n\n\nJ, B = .655, .8\n\ndef grille(duree, dt):\n    if not (math.isfinite(duree) and math.isfinite(dt) and duree>0 and dt>0):\n        raise ValueError("Durée et pas positifs requis")\n    n=round(duree/dt)\n    if n<1 or not math.isclose(n*dt,duree):\n        raise ValueError("La durée doit être un multiple du pas")\n    return [k*dt for k in range(n+1)]\n\ndef simuler_articulation(loi, duree=10., dt=.01, perturbation=lambda t:0., gravite=False, saturation=False):\n    """loi(t,q,w,g,dt) retourne le couple correctif; g est la compensation."""\n    q=w=0.; data=[]\n    for t in grille(duree,dt):\n        g=11.4777*math.cos(q) if gravite else 0.\n        brut=g+loi(t,q,w,g,dt)\n        c=max(-20.,min(20.,brut)) if saturation else brut\n        a=(c-g-B*w+perturbation(t))/J\n        data.append(dict(t=t,q=q,w=w,a=a,c=c,p=c*w,sature=abs(brut-c)>1e-10))\n        q,w=q+dt*w,w+dt*a\n    return data\n\ndef xy(q):\n    a,b=q\n    return (.4*math.cos(a)+.3*math.cos(a+b),.4*math.sin(a)+.3*math.sin(a+b))\n\ndef simuler_mission(loi,T=2.,dt=.005,charge=.5,charge_reelle=.6):\n    """loi(ref,etat,dt) retourne deux couples totaux. Pas de remise à zéro entre segments.\n    Saisie/dépose programmées, sans impact ni collision; masse nominale et réelle distinctes.\n    """\n    qs=postures(); q=qs[0]; w=(0.,0.); data=[]; m=mr=0.\n    for j,(q0,q1) in enumerate(zip(qs,qs[1:]),1):\n        ts=grille(T,dt)\n        for k,t in enumerate(ts):\n            qr,wr,ar=zip(*(profil_cubique(x,y,T,min(t,T)) for x,y in zip(q0,q1)))\n            ref=dict(q=qr,w=wr,a=ar,charge=m,segment=j,t=(j-1)*T+t)\n            brut=tuple(loi(ref,dict(q=q,w=w),dt if k<len(ts)-1 else 0.))\n            c=tuple(max(-lim,min(lim,v)) for v,lim in zip(brut,(20.,8.)))\n            M,h,G=termes_dynamiques(ROBOT,q,w,mr)\n            d=(-.4 if 5<=ref[\'t\']<6 else 0.,0.)\n            z=[c[i]-h[i]-G[i]-(.8,.3)[i]*w[i]+d[i] for i in range(2)]\n            det=M[0][0]*M[1][1]-M[0][1]**2\n            a=((M[1][1]*z[0]-M[0][1]*z[1])/det,(M[0][0]*z[1]-M[0][1]*z[0])/det)\n            data.append(dict(**ref,mesure=q,vitesse=w,acceleration=a,c=c,p=tuple(x*y for x,y in zip(c,w)),sature=any(abs(x-y)>1e-10 for x,y in zip(c,brut)),xy=xy(q),xy_ref=xy(qr)))\n            if k<len(ts)-1: q,w=tuple(x+dt*y for x,y in zip(q,w)),tuple(x+dt*y for x,y in zip(w,a))\n        action=MISSION[j][\'action\']\n        if action==\'saisie\': m,mr=charge,charge_reelle\n        elif action==\'depose\': m=mr=0.\n    return data\n',m.__dict__)

m=types.ModuleType('ex10_mission_asservie'); sys.modules['ex10_mission_asservie']=m
exec('"""RoboLab S5 — EX10 — corrigé enseignant."""\nimport math\nfrom support_s4 import grille, simuler_articulation, simuler_mission, couples, ROBOT\n\ndef creer_commande(mode, Kp=12., Ki=2., Kd=4.):\n    if mode not in (\'anticipation\',\'pid\',\'combine\'): raise ValueError(\'Mode inconnu\')\n    I=[0.,0.]\n    def loi(ref,etat,dt):\n        ff=couples(ROBOT,ref[\'q\'],ref[\'w\'],ref[\'a\'],ref[\'charge\']) if mode!=\'pid\' else (0.,0.)\n        if mode==\'anticipation\': return ff\n        c=[]\n        for i,lim in enumerate((20,8)):\n            e=ref[\'q\'][i]-etat[\'q\'][i]; de=ref[\'w\'][i]-etat[\'w\'][i]\n            trial=I[i]+dt*e; raw=ff[i]+Kp*e+Ki*trial+Kd*de\n            if not ((raw>lim and e>0) or (raw< -lim and e<0)): I[i]=trial\n            c.append(ff[i]+Kp*e+Ki*I[i]+Kd*de)\n        return tuple(c)\n    return loi\n\ndef bilan(data):\n    def integ(v): return sum((x+y)/2*(b[\'t\']-a[\'t\']) for x,y,a,b in zip(v,v[1:],data,data[1:]))\n    T=data[-1][\'t\']-data[0][\'t\']\n    e=[[math.degrees(r[\'q\'][i]-r[\'mesure\'][i]) for r in data] for i in range(2)]\n    xy=[1000*math.dist(r[\'xy\'],r[\'xy_ref\']) for r in data]\n    maxima={k:[max(abs(r[k][i]) for r in data) for i in range(2)] for k in (\'c\',\'vitesse\',\'acceleration\',\'p\')}\n    return dict(erreur_max_deg=[max(map(abs,v)) for v in e],erreur_rms_deg=[math.sqrt(integ([x*x for x in v])/T) for v in e],xy_max_mm=max(xy),xy_rms_mm=math.sqrt(integ([x*x for x in xy])/T),maxima=maxima,admissible=all(x<=l for k,lims in zip(maxima,((20/1.5,8/1.5),(2,3),(5,8),(40,24))) for x,l in zip(maxima[k],lims)),saturation_s=sum(b[\'t\']-a[\'t\'] for a,b in zip(data,data[1:]) if a[\'sature\']),travail_positif=[integ([max(0,r[\'p\'][i]) for r in data]) for i in range(2)],travail_negatif=[integ([min(0,r[\'p\'][i]) for r in data]) for i in range(2)])\n\nif __name__ == "__main__":\n    loi=creer_commande(\'anticipation\')\n    ref=dict(q=(0,0),w=(0,0),a=(0,0),charge=.5)\n    assert all(math.isclose(x,y) for x,y in zip(loi(ref,dict(q=(0,0),w=(0,0)),.005),(11.4777,3.2373)))\n    print("Tests de base réussis")\n',m.__dict__)

m=types.ModuleType('ex11_optimisation'); sys.modules['ex11_optimisation']=m
exec('"""Optimisation reproductible, modèle S5 inchangé hors commande et profil."""\nimport math,itertools\nimport support_s4 as s\nfrom ex10_mission_asservie import bilan,creer_commande\n\ndef quintique(x,y,T,t):\n    u=t/T; d=y-x\n    return x+d*(10*u**3-15*u**4+6*u**5),d/T*(30*u**2-60*u**3+30*u**4),d/T**2*(60*u-180*u**2+120*u**3)\n\nCUBIQUE=s.profil_cubique\n\ndef commande(gains, friction=True):\n    I=[0.,0.]\n    def loi(ref,etat,dt):\n        ff=s.couples(s.ROBOT,ref[\'q\'],ref[\'w\'],ref[\'a\'],ref[\'charge\']); out=[]\n        for i,(kp,ki,kd) in enumerate(gains):\n            f=ff[i]+((.8,.3)[i]*ref[\'w\'][i] if friction else 0)\n            e=ref[\'q\'][i]-etat[\'q\'][i]; de=ref[\'w\'][i]-etat[\'w\'][i]; lim=(20,8)[i]\n            trial=I[i]+dt*e; raw=f+kp*e+ki*trial+kd*de\n            if not ((raw>lim and e>0) or (raw< -lim and e<0)): I[i]=trial\n            out.append(f+kp*e+ki*I[i]+kd*de)\n        return tuple(out)\n    return loi\n\ndef sim(gains,T=2,dt=.005,profil=\'quintique\',friction=True,charge=.6):\n    old=s.profil_cubique\n    try:\n        s.profil_cubique=quintique if profil==\'quintique\' else CUBIQUE\n        return s.simuler_mission(commande(gains,friction),T,dt,charge_reelle=charge)\n    finally: s.profil_cubique=old\n\ndef evaluer(data):\n    b=bilan(data); b[\'duree\']=data[-1][\'t\']; b[\'points_mm\']=[1000*math.dist(r[\'xy\'],r[\'xy_ref\']) for i,r in enumerate(data) if i==len(data)-1 or data[i+1][\'segment\']!=r[\'segment\']]; b[\'valide\']=b[\'admissible\'] and b[\'xy_rms_mm\']<=10 and b[\'xy_max_mm\']<=20 and max(b[\'points_mm\'][1],b[\'points_mm\'][4])<=10\n    return b\n\ndef recherche():\n    results=[]\n    for T,profil,g1,g2 in itertools.product((1.5,2.,2.5,3.),(\'cubique\',\'quintique\'),((12,2,4),(24,4,6),(40,6,8)),((8,1,2),(16,3,3),(24,4,4))):\n        gains=(g1,g2); b=evaluer(sim(gains,T,.01,profil)); results.append(dict(T=T,profil=profil,gains=gains,**b))\n    return sorted(results,key=lambda b:(not b[\'valide\'],b[\'T\'] if b[\'valide\'] else b[\'xy_rms_mm\'],b[\'xy_rms_mm\']))\n\nif __name__==\'__main__\':\n    import json\n    r=recherche(); print(\'valides\',sum(x[\'valide\'] for x in r)); print(r[:3]); print(\'baseline\',evaluer(s.simuler_mission(creer_commande(\'combine\')))); print(\'fin\',evaluer(sim(r[0][\'gains\'],r[0][\'T\'],.0025,r[0][\'profil\']))); open(\'recherche.json\',\'w\').write(json.dumps(r,indent=2))\n',m.__dict__)
from ex11_optimisation import *
import matplotlib.pyplot as plt


In [2]:
assert quintique(0,1,2,0)==(0,0,0)
assert quintique(0,1,2,2)==(1,0,0)
r=recherche()
print(len(r),sum(x["valide"] for x in r))
print(r[0])

72 27
{'T': 1.5, 'profil': 'cubique', 'gains': ((40, 6, 8), (24, 4, 4)), 'erreur_max_deg': [0.972526184385531, 0.6213466549665277], 'erreur_rms_deg': [0.4206482053339592, 0.2487020499906405], 'xy_max_mm': 9.692599691545684, 'xy_rms_mm': 4.568032511276723, 'maxima': {'c': [10.703010473175823, 2.4647513179329175], 'vitesse': [1.075408133427405, 0.6951034268336805], 'acceleration': [2.892544787930885, 7.261464660680878], 'p': [6.547356963188397, 1.2336184719743073]}, 'admissible': True, 'saturation_s': 0, 'travail_positif': [13.656521884162794, 1.3767820700236348], 'travail_negatif': [-9.979576806639619, -1.973930428338069], 'duree': 10.5, 'points_mm': [0.5203396892939468, 0.24662927327202633, 7.183836145383161, 4.414771999667549, 3.2718220476309106, 3.141082467513801, 0.3038597711698962], 'valide': True}


Le cubique minimise la RMS sur la grille à durée minimale. On retient aussi le quintique pour sa marge en accélération. Comparaison à pas affiné et test de sensibilité à la charge.

In [3]:
g=r[0]["gains"]
base=s.simuler_mission(creer_commande("combine"),dt=.0025)
scenarios={"Initial":base, **{p:sim(g,1.5,.0025,p) for p in ("cubique","quintique")}}
for name,data in scenarios.items():
 b=evaluer(data); print(name,b)
for dt in (.005,.0025,.00125):
 b=evaluer(sim(g,1.5,dt)); print(dt,b["xy_rms_mm"],b["maxima"]["acceleration"]); assert b["valide"]
for m in (.5,.6,.7):
 b=evaluer(sim(g,1.5,.0025,charge=m)); print(m,b["xy_rms_mm"],b["valide"])

Initial {'erreur_max_deg': [5.831996996833556, 2.41164444394914], 'erreur_rms_deg': [2.8596819634603614, 1.0942360737698125], 'xy_max_mm': 67.39046334227446, 'xy_rms_mm': 30.188078871007917, 'maxima': {'c': [11.155737153971291, 2.638099228124289], 'vitesse': [0.7797475667665036, 0.5390502352324729], 'acceleration': [2.299770724372475, 9.063822721052336], 'p': [4.897194998059285, 1.1247859171583252]}, 'admissible': False, 'saturation_s': 0, 'travail_positif': [13.036646323713384, 1.4426623170861592], 'travail_negatif': [-9.45906162866815, -2.301602852140412], 'duree': 14.0, 'points_mm': [3.910036281527947, 3.471621033803354, 67.12352774983947, 15.388921234029079, 28.77328053161529, 10.14711084695982, 20.825116298151006], 'valide': False}
cubique {'erreur_max_deg': [0.9705689784789752, 0.560403899118083], 'erreur_rms_deg': [0.42052251645287647, 0.2416249677750543], 'xy_max_mm': 9.598247554599226, 'xy_rms_mm': 4.541590268241974, 'maxima': {'c': [10.695455815079498, 2.461232788984328], 'vi

0.00125 4.551342426859144 [2.7501810162660276, 5.026270836499002]
0.5 1.8036413044174184 True
0.6 4.5577223055107705 True


0.7 8.39193165828836 False


In [4]:
for key,unit in [("erreur","mm"),("acceleration","rad/s²"),("c","N·m"),("vitesse","rad/s"),("p","W")]:
 fig,ax=plt.subplots(figsize=(10,4))
 for name,data in scenarios.items():
  t=[v["t"] for v in data]
  if key=="erreur": ax.plot(t,[1000*math.dist(v["xy"],v["xy_ref"]) for v in data],label=name)
  else: ax.plot(t,[v[key][1] for v in data],label=name)
 ax.set(xlabel="Temps (s)",ylabel=unit,title=key+ (" — coude" if key!="erreur" else " cartésienne")); ax.legend(); ax.grid(); plt.show()

/tmp/ipykernel_2700/4146289182.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set(xlabel="Temps (s)",ylabel=unit,title=key+ (" — coude" if key!="erreur" else " cartésienne")); ax.legend(); ax.grid(); plt.show()
